# 🏎️ APEX AI — Production Fine-Tuning & Live Tunnel Server (v2.2)

This notebook delivers an end-to-end, zero-error pipeline on **Google Colab's Free T4 GPU**:
1. **Fine-tunes Llama-3-8B** using **Unsloth (QLoRA 4-bit)** on the updated F1 Senior Technical Director dataset.
2. **Eliminates Truncation & Word Limits**: `max_new_tokens=850` with repetition penalties and anti-looping safeguards.
3. **Forces Definitive Technical Conclusions**: Telemetry numbers, tyre degradation rates, aero operating windows, and closing summaries.
4. **Launches a Live Ngrok Tunnel**: Directly bridges your local `APEX CLI` terminal to Colab.

---

## Step 1: Install Dependencies (Fast 4-bit Unsloth Stack)

In [1]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes triton
!pip install -q fastapi uvicorn pyngrok nest-asyncio

## Step 2: Load 4-bit Llama-3-8B & Attach LoRA Adapters

In [4]:
# Load 4-bit Llama-3-8B with Unsloth
from unsloth import FastLanguageModel
import torch
import os

max_seq_length = 2048
dtype = None
load_in_4bit = True

# -----------------------------------------------------------------------------
# FAST RELOAD OPTION: If you previously saved your model to Google Drive,
# change LOAD_FROM_DRIVE = True to skip Steps 3 and 4 entirely (~30 sec boot!)
# -----------------------------------------------------------------------------
LOAD_FROM_DRIVE = False
drive_path = "/content/drive/MyDrive/apex_ai_lora"

if LOAD_FROM_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print(f"Loading fine-tuned APEX AI model from Google Drive: {drive_path}...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = drive_path,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model)
    model.eval()
    print("\n" + "="*70)
    print("✓ Model loaded directly from Google Drive in ~30 seconds!")
    print("👉 You can now SKIP Steps 3 & 4 and jump straight to Step 5 (Run Server).")
    print("="*70 + "\n")
else:
    # Standard initial setup: Load base Llama-3 and attach LoRA adapters
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 42,
    )
    print("✓ Base model and LoRA adapters initialized successfully! Proceed to Step 3.")


==((====))==  Unsloth 2026.11.3: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).
✓ Model and LoRA adapters initialized successfully!


## Step 3: Pull Latest Clean Training Dataset from GitHub

In [ ]:
# Pull latest clean dataset directly from master
import os
import urllib.request
from datasets import load_dataset

dataset_url = "https://raw.githubusercontent.com/bhxvxsh/yeah-def-my-nyc-project/master/03_Training_Dataset/f1_technical_alpaca.json"
local_file = "f1_technical_alpaca.json"

print("Downloading technical training dataset from GitHub master...")
urllib.request.urlretrieve(dataset_url, local_file)

# Cleanly refresh repo without divergent branch conflicts
!rm -rf repo && git clone --depth 1 https://github.com/bhxvxsh/yeah-def-my-nyc-project.git repo

if os.path.exists(local_file):
    data_path = local_file
elif os.path.exists("repo/03_Training_Dataset/f1_technical_alpaca.json"):
    data_path = "repo/03_Training_Dataset/f1_technical_alpaca.json"
else:
    data_path = dataset_url

print(f"✓ Loading verified technical dataset from: {data_path}")
dataset = load_dataset("json", data_files=data_path, split="train")

prompt_template = """Below is an instruction from a Formula 1 team principal or research analyst. Provide an uncompromisingly technical, authoritative, and complete engineering evaluation. Do NOT output generic python feature templates. Provide real telemetry benchmarks, degradation numbers, and always conclude with a definitive technical summary.

### Instruction:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = prompt_template.format(instruction, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

formatted_dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"✓ Loaded {len(dataset)} verified technical F1 samples.")


## Step 4: Fine-Tune Model (Quick ~10-15 Min Run)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()
print("✓ Fine-tuning complete! Activating fast inference mode...")
FastLanguageModel.for_inference(model)
model.eval()

## Step 4b (Recommended): Save Fine-Tuned Model to Google Drive

Run this cell **once** after Step 4 completes. It will mount your Google Drive and save the fine-tuned LoRA adapters (~50 MB, takes ~5 seconds).

> **Benefit**: In future Colab sessions, you can set `LOAD_FROM_DRIVE = True` in Step 2 to boot in **30 seconds** without retraining!

In [ ]:
# Mount Google Drive and save LoRA adapter weights
from google.colab import drive
drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/apex_ai_lora"
print(f"Saving fine-tuned APEX AI model to: {save_dir}...")

# Save LoRA adapter weights & tokenizer (PEFT / Unsloth native)
try:
    model.save_pretrained(save_dir)
except AttributeError:
    model.save_pretrained_merged(save_dir, tokenizer, save_method="lora")

tokenizer.save_pretrained(save_dir)

print("\n" + "="*70)
print(f"✓ SUCCESS: Model permanently saved to Google Drive at: {save_dir}")
print("👉 Next time you open Colab:")
print("   1. Run Step 1 (Install Dependencies).")
print("   2. In Step 2, set LOAD_FROM_DRIVE = True.")
print("   3. Jump straight to Step 5 to start your API server! (Zero retraining needed)")
print("="*70 + "\n")


## Step 5: Start Live Fast-API Server & Ngrok Tunnel for APEX CLI

> **Important:** Set your free Ngrok token below if you have one (`ngrok.set_auth_token("YOUR_TOKEN")`). If not, the public tunnel will start directly.

In [ ]:
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
from pyngrok import ngrok
import traceback

# Ensure model is in inference mode
FastLanguageModel.for_inference(model)
model.eval()

app = FastAPI(title="APEX AI Inference Server")

class QueryRequest(BaseModel):
    query: str

@app.get("/health")
def health():
    return {"status": "ok", "model": "APEX-AI-Senior-Race-Strategist"}

@app.post("/ask")
def ask(req: QueryRequest):
    try:
        user_text = req.query.strip()
        print(f"\n[User Query Received]: {user_text}")
        
        prompt = (
            "Below is an instruction from a Formula 1 team principal or research analyst. "
            "Provide an uncompromisingly technical, authoritative, and complete engineering evaluation. "
            "Do NOT output generic python feature templates. Provide real telemetry benchmarks, degradation numbers, "
            "and always conclude with a definitive technical summary.\n\n"
            f"### Instruction:\n{user_text}\n\n"
            "### Response:\n"
        )
        
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        with torch.inference_mode():
            outputs = model.generate(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=850,        # Increased token ceiling: eliminates truncation
                temperature=0.6,           # Controlled, factual outputs
                top_p=0.9,
                repetition_penalty=1.15,   # Prevents looping on lists of features
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
        answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        
        print(f"[Generated Response Length]: {len(answer)} chars")
        return {"response": answer}
        
    except Exception as e:
        err_msg = f"{type(e).__name__}: {str(e)}\n{traceback.format_exc()}"
        print(f"[ERROR]: {err_msg}")
        return {"response": f"Colab Server Error: {str(e)}"}

# Optional: Set your token if you have one
# ngrok.set_auth_token("YOUR_FREE_NGROK_TOKEN")

# Kill any previous sessions
ngrok.kill()
public_url = ngrok.connect(8000).public_url

print("\n" + "=" * 70)
print(f"🚀 YOUR LIVE APEX AI COLAB URL: {public_url}")
print(f"👉 In your local APEX CLI, type: /connect {public_url}")
print("=" * 70 + "\n")

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)